# CAC40 Volatility Surface — Unified Modelling Pipeline
**Date:** 12 February 2025 | **Underlying:** CAC40 (EURONEXT)  
**Models:** Implied Vol Surface → SVI Calibration → Dupire Local Vol → Heston Multi-Maturity Calibration → Exotic Pricing

---

### Architecture
```
Market Data (CAC40 options + EURIBOR6M)
    │
    ├── Forward Bootstrap (call-put parity, piecewise dividend yields)
    │
    ├── Implied Vol Surface (OTM Newton-Raphson, arbitrage checks)
    │
    ├── SVI Calibration (per-slice, arbitrage-free, DE + L-BFGS-B)
    │
    ├── Dupire Local Vol (analytical from SVI derivatives)
    │
    ├── Heston Calibration (multi-maturity, vectorised slice pricer, LM)
    │
    └── Exotic Pricing (MC: Local Vol QE vs Heston, barrier/asian/european)
```


## 0. Imports & Configuration

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/home/claude/vol_project")  # adjust to your path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D

# Internal modules
from core.market_data    import load_options, load_rates, collect_by_expiry, S0, UNDERLYING
from core.forward_curve  import run_forward_bootstrap
from core.implied_vol    import build_iv_surface, check_calendar_arbitrage, check_butterfly_arbitrage
from core.interpolation  import calibrate_svi_surface, svi_surface_iv, svi_iv
from models.local_vol    import build_local_vol_surface, mc_local_vol, price_european_lv, price_barrier_lv, price_asian_lv
from models.heston       import calibrate_heston, mc_heston, heston_implied_vol

# Paths — update to your data location
OPT_PATH  = "/mnt/user-data/uploads/CAC40_MarketOptions_12022025__1_.csv"
RATE_PATH = "/mnt/user-data/uploads/EURIBOR6M_ZCRates_12022025__1_.csv"

STYLE = {
    "figure.facecolor": "white", "axes.facecolor": "#f9f9f9",
    "axes.grid": True, "grid.alpha": 0.4, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
}
plt.rcParams.update(STYLE)
COLORS = plt.cm.plasma(np.linspace(0.1, 0.9, 13))
print("All modules loaded ✓")


## 1. Market Data Loading

In [ ]:
df_options = load_options(OPT_PATH)
zc_rate, df_rates = load_rates(RATE_PATH)
data = collect_by_expiry(df_options)
expiries = data["expiries"]

print(f"Underlying  : {UNDERLYING}")
print(f"Spot price  : {S0}")
print(f"Maturities  : {len(expiries)} slices")
print(f"  from T = {expiries[0]:.4f}y to T = {expiries[-1]:.4f}y")
print(f"Strikes     : {sum(len(data['strikes'][T]) for T in expiries)} total market quotes")

# Plot EURIBOR6M ZC curve
fig, ax = plt.subplots(figsize=(9, 4))
t_ax = np.linspace(0.02, 5.5, 300)
ax.plot(t_ax, zc_rate(t_ax)*100, color="#2563eb", lw=2.5, label="Cubic spline interpolation")
ax.scatter(df_rates["Expiry"], df_rates["ZCRate"]*100, color="#dc2626", zorder=5, s=50, label="Market quotes")
ax.set_title("EURIBOR6M Zero-Coupon Rate Curve  |  12-Feb-2025", fontweight="bold")
ax.set_xlabel("Maturity (years)"); ax.set_ylabel("ZC Rate (%)")
ax.legend(); plt.tight_layout(); plt.show()


## 2. Forward Bootstrap & Dividend Curve

In [ ]:
implied_fwds, divs, div_func, fwd_curve = run_forward_bootstrap(data, zc_rate, S0)

print("Implied Forwards (via Call-Put Parity Bisection):")
for T, F in implied_fwds.items():
    d = divs[T]*100
    print(f"  T={T:.4f}y  F={F:.2f}  q={d:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
t_ax = np.linspace(0.01, expiries[-1]+0.3, 600)

axes[0].plot(t_ax, fwd_curve(t_ax), "#2563eb", lw=2.5, label="Smooth forward curve")
axes[0].scatter(list(implied_fwds.keys()), list(implied_fwds.values()), 
                color="#dc2626", s=60, zorder=5, label="Implied forwards")
axes[0].axhline(S0, color="gray", ls="--", lw=1.2, label=f"Spot = {S0}")
axes[0].set_title("CAC40 Implied Forward Curve", fontweight="bold")
axes[0].set_xlabel("Maturity (years)"); axes[0].set_ylabel("Price"); axes[0].legend()

axes[1].plot(t_ax, div_func(t_ax)*100, "#16a34a", lw=2.5, drawstyle="steps-post")
axes[1].scatter(list(divs.keys()), [v*100 for v in divs.values()], color="#dc2626", s=60, zorder=5)
axes[1].set_title("Piecewise-Constant Dividend Yield Curve\n(seasonal spike in Jun/Jul → CAC40 dividend season)", fontweight="bold")
axes[1].set_xlabel("Maturity (years)"); axes[1].set_ylabel("Dividend Yield (%)")
plt.tight_layout(); plt.show()


## 3. Implied Volatility Surface

OTM options are selected for each strike (call if K > F, put if K < F) to minimise
bid-ask spread effects. Implied vols are extracted via Newton-Raphson with Halley
acceleration, falling back to Brent's method for degenerate cases.


In [ ]:
iv_surface = build_iv_surface(data, fwd_curve, zc_rate, use_otm=True)

# Arbitrage checks
cal_viol = check_calendar_arbitrage(iv_surface)
but_viol = check_butterfly_arbitrage(iv_surface)
print(f"Calendar spread violations : {len(cal_viol)}")
print(f"Butterfly spread violations: {len(but_viol)}")

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()
for idx, T in enumerate(expiries):
    ax = axes[idx]
    surf = iv_surface[T]; y = surf["moneyness"]; iv = surf["ivs"]; v = surf["valid"]
    ax.scatter(y[v]*100, iv[v]*100, color=COLORS[idx], s=35, zorder=5)
    ax.set_title(f"T = {T:.3f}y", fontsize=9); ax.set_ylim(0, 40)
    ax.set_xlabel("Log-moneyness (%)", fontsize=7); ax.set_ylabel("IV (%)", fontsize=7)
for idx in range(len(expiries), len(axes)): axes[idx].set_visible(False)
fig.suptitle("CAC40 Implied Volatility Smiles  |  12-Feb-2025", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


## 4. SVI Calibration

SVI raw parameterisation (Gatheral 2004):
$$w(y) = a + b\left(\rho(y-m) + \sqrt{(y-m)^2 + \sigma^2}\right)$$

where $w = \sigma_{BS}^2 \cdot T$ (total variance) and $y = \ln(K/F)$ (log-moneyness).

Calibration: differential evolution (global) → L-BFGS-B (local), weighted by ATM proximity.  
Butterfly-free constraint: $b(1+|\rho|) \leq 4$.


In [ ]:
svi_params = calibrate_svi_surface(iv_surface)

print("SVI Parameters per slice:")
print(f"{'T':>8}  {'a':>9}  {'b':>9}  {'rho':>8}  {'m':>8}  {'sigma':>8}")
for T, p in svi_params.items():
    print(f"  {T:6.4f}  {p[0]:9.5f}  {p[1]:9.5f}  {p[2]:8.4f}  {p[3]:8.4f}  {p[4]:8.4f}")

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()
y_fine = np.linspace(-0.4, 0.4, 200)
for idx, T in enumerate(expiries):
    if T not in svi_params: continue
    ax = axes[idx]
    surf = iv_surface[T]; v = surf["valid"]
    ax.scatter(surf["moneyness"][v]*100, surf["ivs"][v]*100, c="black", s=25, zorder=5, label="Market")
    ax.plot(y_fine*100, svi_iv(y_fine, T, *svi_params[T])*100, color=COLORS[idx], lw=2.5, label="SVI")
    # Compute slice RMSE
    y_mkt = surf["moneyness"][v]; iv_mkt = surf["ivs"][v]
    iv_svi_mkt = svi_iv(y_mkt, T, *svi_params[T])
    rmse = np.sqrt(np.mean((iv_svi_mkt - iv_mkt)**2)) * 100
    ax.set_title(f"T={T:.3f}y  RMSE={rmse:.2f}%", fontsize=9); ax.set_ylim(0, 40)
    ax.set_xlabel("Log-moneyness (%)", fontsize=7); ax.set_ylabel("IV (%)", fontsize=7)
    if idx == 0: ax.legend(fontsize=7)
for idx in range(len(expiries), len(axes)): axes[idx].set_visible(False)
fig.suptitle("SVI Calibration vs Market  |  CAC40", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


## 5. Full Implied Volatility Surface (3D + Heatmap)

In [ ]:
y_fine  = np.linspace(-0.45, 0.45, 120)
T_fine  = np.linspace(expiries[0], expiries[-1], 100)
IV_surf = np.array([[svi_surface_iv(y, T, svi_params)*100 for y in y_fine] for T in T_fine])
YY, TT  = np.meshgrid(y_fine, T_fine)

fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection="3d")
surf_plot = ax.plot_surface(YY*100, TT, IV_surf, cmap="plasma", alpha=0.88)
ax.set_xlabel("Log-moneyness (%)", labelpad=10)
ax.set_ylabel("Maturity (years)", labelpad=10)
ax.set_zlabel("Implied Vol (%)", labelpad=10)
ax.set_title("CAC40 Implied Volatility Surface  |  12-Feb-2025", fontweight="bold", pad=20)
fig.colorbar(surf_plot, ax=ax, shrink=0.4, label="IV (%)")
ax.view_init(elev=25, azim=-60)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.contourf(y_fine*100, T_fine, IV_surf, levels=30, cmap="plasma")
ax.contour(y_fine*100, T_fine, IV_surf, levels=10, colors="white", linewidths=0.4, alpha=0.5)
fig.colorbar(im, ax=ax, label="IV (%)")
ax.set_xlabel("Log-moneyness (%)"); ax.set_ylabel("Maturity (years)")
ax.set_title("Implied Volatility Surface — Heatmap", fontweight="bold")
plt.tight_layout(); plt.show()


## 6. Dupire Local Volatility Surface

Local vol is derived analytically from the SVI surface using the Dupire formula
in total variance / log-moneyness coordinates (Gatheral, *The Volatility Surface*, Ch.1):

$$\sigma_{loc}^2(y,T) = \frac{\partial_T w}{1 - \frac{y}{w}\partial_y w + \frac{1}{4}\left(-\frac{1}{4} - \frac{1}{w} + \frac{y^2}{w^2}\right)(\partial_y w)^2 + \frac{1}{2}\partial_{yy}w}$$

**Key property:** $\sigma_{loc}^2(K,T) = \mathbb{E}[v_T | S_T = K]$ — the local vol is the conditional
expectation of the instantaneous variance. This means it prices vanilla options exactly by 
construction, but its dynamic smile behaviour differs from stochastic vol models.


In [ ]:
y_grid, T_lv_grid, lv_surf, lv_func = build_local_vol_surface(
    svi_params, y_min=-0.35, y_max=0.35, n_y=80, T_min=0.15, T_max=expiries[-1], n_T=80)

lv_disp = np.clip(lv_surf, 0, 0.50)
YY_lv, TT_lv = np.meshgrid(y_grid, T_lv_grid)

fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection="3d")
surf_plot = ax.plot_surface(YY_lv*100, TT_lv, lv_disp.T*100, cmap="viridis", alpha=0.88)
ax.set_xlabel("Log-moneyness (%)", labelpad=10)
ax.set_ylabel("Maturity (years)", labelpad=10)
ax.set_zlabel("Local Vol (%)", labelpad=10)
ax.set_title("CAC40 Dupire Local Volatility Surface  |  12-Feb-2025", fontweight="bold", pad=20)
fig.colorbar(surf_plot, ax=ax, shrink=0.4, label="LV (%)")
ax.view_init(elev=25, azim=-60)
plt.tight_layout(); plt.show()

# LV vs IV at selected maturities
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, T_target in enumerate([0.35, 1.0, 2.0]):
    T_comp = min(svi_params.keys(), key=lambda t: abs(t - T_target))
    iv_vals = np.array([svi_surface_iv(y, T_comp, svi_params)*100 for y in y_grid])
    lv_vals = np.array([lv_func(y, T_comp)*100 for y in y_grid])
    axes[idx].plot(y_grid*100, iv_vals, "#2563eb", lw=2.5, label=f"Implied Vol")
    axes[idx].plot(y_grid*100, lv_vals, "#dc2626", lw=2.5, label=f"Local Vol")
    axes[idx].set_title(f"T ≈ {T_comp:.2f}y", fontweight="bold")
    axes[idx].set_xlabel("Log-moneyness (%)"); axes[idx].set_ylabel("Vol (%)")
    axes[idx].legend()
    # IV vs LV relationship
    axes[idx].annotate("LV < IV in wings\n(smile flattens under LV dynamics)",
                       xy=(0.05, 0.95), xycoords="axes fraction", ha="right", va="top",
                       fontsize=8, color="gray")
fig.suptitle("Implied Vol vs Dupire Local Vol  |  LV flattens the forward smile", 
             fontweight="bold", fontsize=12)
plt.tight_layout(); plt.show()


## 7. Heston Model Calibration

$$dS_t = (r-q)S_t\,dt + \sqrt{v_t}S_t\,dW^1_t$$
$$dv_t = \kappa(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW^2_t, \quad dW^1 dW^2 = \rho\,dt$$

**Parameters:** mean-reversion speed $\kappa$, long-term variance $\theta$, vol-of-vol $\xi$, 
correlation $\rho$, initial variance $v_0$.

**Pricing:** Semi-closed form via P1/P2 dual integration of the characteristic function.  
**Calibration:** Multi-maturity global least-squares (Levenberg-Marquardt) with ATM-weighted
residuals and Feller condition monitoring ($2\kappa\theta > \xi^2$).


In [ ]:
# Build SVI-smoothed IV surface as calibration targets
iv_surface_clean = {}
for T in expiries:
    if T not in svi_params: continue
    surf = iv_surface[T]; K_arr = surf["strikes"]; F = surf["forward"]; y_arr = np.log(K_arr/F)
    iv_svi = svi_iv(y_arr, T, *svi_params[T])
    iv_surface_clean[T] = {"strikes": K_arr, "ivs": iv_svi, "moneyness": y_arr,
        "forward": F, "rate": surf["rate"], "valid": np.ones(len(K_arr), dtype=bool)}

heston_result = calibrate_heston(S0, iv_surface_clean, fwd_curve, zc_rate,
    x0=np.array([2.0, 0.04, 0.30, -0.7, 0.04]), verbose=True)

hparams = heston_result["params"]
model_ivs = heston_result["model_ivs"]


In [ ]:
# Heston smile vs market — 6 selected maturities
sel_T = expiries[::2][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, T in enumerate(sel_T):
    ax = axes[idx]
    surf = iv_surface[T]; K_arr = surf["strikes"]; F = surf["forward"]
    y_arr = np.log(K_arr/F)*100

    ax.scatter(y_arr, surf["ivs"]*100, c="black", s=40, zorder=5, label="Market")
    ax.plot(y_arr, iv_surface_clean[T]["ivs"]*100, "#2563eb", lw=1.5, ls="--", label="SVI")
    ax.plot(y_arr, model_ivs[T]*100, "#dc2626", lw=2.5, label="Heston")
    ax.set_title(f"T = {T:.3f}y", fontweight="bold"); ax.set_xlabel("Log-moneyness (%)")
    ax.set_ylabel("IV (%)"); ax.set_ylim(0, 40); ax.legend(fontsize=8)

rmse = heston_result['rmse']*100
p = hparams
fig.suptitle(f"Heston Calibration  |  RMSE = {rmse:.3f} vol pts  |  "
             f"κ={p['kappa']:.2f}  θ={p['theta']:.4f}  ξ={p['xi']:.3f}  ρ={p['rho']:.3f}  v₀={p['v0']:.4f}",
             fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


## 8. Monte Carlo Simulation: Local Vol vs Heston

- **Local Vol MC:** Euler-Maruyama in log-space with drift from fwd curve  
- **Heston MC:** Quadratic Exponential (Andersen 2008) scheme for variance,
  exact Milstein-style log-price update


In [ ]:
T_mc = min(expiries, key=lambda t: abs(t - 1.0))
r_mc = float(zc_rate(T_mc))
F_mc = float(fwd_curve(T_mc))
q_mc = max(r_mc - np.log(F_mc/S0)/T_mc, 0.)

print(f"MC parameters: T={T_mc:.3f}y, r={r_mc:.4f}, q={q_mc:.4f}, F={F_mc:.2f}")

N_steps, M_paths = 252, 30_000
print(f"Running {M_paths:,} paths × {N_steps} steps...")

paths_lv = mc_local_vol(S0, T_mc, fwd_curve, zc_rate, lv_func,
                         N=N_steps, M=M_paths, seed=42)
paths_heston, v_heston = mc_heston(S0, T_mc, r_mc, q_mc,
                                    **hparams, N=N_steps, M=M_paths, scheme="QE", seed=42)
print("Done ✓")


In [ ]:
t_plot = np.linspace(0, T_mc, N_steps+1)
n_show = 25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i in range(n_show):
    axes[0].plot(t_plot, paths_lv[i], alpha=0.35, lw=0.8, color="#2563eb")
axes[0].axhline(S0, color="black", ls="--", lw=1.2, label=f"S₀={S0}")
axes[0].set_title(f"Local Vol MC Paths  ({n_show} shown)", fontweight="bold")
axes[0].set_xlabel("Time (years)"); axes[0].set_ylabel("CAC40 Level"); axes[0].legend()

for i in range(n_show):
    axes[1].plot(t_plot, paths_heston[i], alpha=0.35, lw=0.8, color="#dc2626")
axes[1].axhline(S0, color="black", ls="--", lw=1.2)
axes[1].set_title(f"Heston MC Paths  ({n_show} shown)", fontweight="bold")
axes[1].set_xlabel("Time (years)"); axes[1].set_ylabel("CAC40 Level")
plt.tight_layout(); plt.show()

# Heston variance paths — volatility clustering
fig, ax = plt.subplots(figsize=(11, 4))
for i in range(8):
    ax.plot(t_plot, np.sqrt(v_heston[i])*100, alpha=0.7, lw=1.5)
ax.axhline(np.sqrt(hparams['theta'])*100, color="black", ls="--", lw=1.5, 
           label=f"LT vol θ = {np.sqrt(hparams['theta'])*100:.1f}%")
ax.set_title("Heston: Stochastic Volatility Paths (vol clustering visible)", fontweight="bold")
ax.set_xlabel("Time (years)"); ax.set_ylabel("Inst. Vol (%)"); ax.legend()
plt.tight_layout(); plt.show()


## 9. Exotic Options Pricing: BS vs Local Vol vs Heston

Three product types compared:
- **European Call** (ATM): tests model calibration accuracy
- **Up-and-Out Barrier Call** (10% OTM barrier): strongly path-dependent, sensitive to vol dynamics
- **Asian Call** (arithmetic average): sensitive to variance term structure

**Key intuition:** Barrier options are most sensitive to the *forward smile* shape —
Local Vol and Heston can give very different answers despite matching the current vanilla surface.


In [ ]:
from core.implied_vol import bs_call
from core.interpolation import svi_surface_iv

K_atm = round(F_mc / 50) * 50
barrier_up = K_atm * 1.10
sigma_flat = float(svi_surface_iv(0.0, T_mc, svi_params))

print(f"ATM K={K_atm}  |  Barrier={barrier_up}  |  ATM IV={sigma_flat*100:.2f}%")

# BS flat vol MC
paths_bs = mc_local_vol(S0, T_mc, fwd_curve, zc_rate,
    lambda y, t: sigma_flat, N=N_steps, M=M_paths, seed=42)

# Prices
euro_bs     = price_european_lv(paths_bs,     K_atm, T_mc, r_mc, "call")
euro_lv     = price_european_lv(paths_lv,     K_atm, T_mc, r_mc, "call")
euro_heston = price_european_lv(paths_heston, K_atm, T_mc, r_mc, "call")

bar_bs      = price_barrier_lv(paths_bs,     K_atm, barrier_up, T_mc, r_mc, "up-and-out", "call")
bar_lv      = price_barrier_lv(paths_lv,     K_atm, barrier_up, T_mc, r_mc, "up-and-out", "call")
bar_heston  = price_barrier_lv(paths_heston, K_atm, barrier_up, T_mc, r_mc, "up-and-out", "call")

asian_bs     = price_asian_lv(paths_bs,     K_atm, T_mc, r_mc, "call")
asian_lv     = price_asian_lv(paths_lv,     K_atm, T_mc, r_mc, "call")
asian_heston = price_asian_lv(paths_heston, K_atm, T_mc, r_mc, "call")

df_res = pd.DataFrame({
    "Product":      ["European Call", "Up-and-Out Barrier", "Asian Call (Arith.)"],
    "BS (flat ATM)": [f"{euro_bs:.2f}",  f"{bar_bs:.2f}",  f"{asian_bs:.2f}"],
    "Local Vol":    [f"{euro_lv:.2f}",   f"{bar_lv:.2f}",  f"{asian_lv:.2f}"],
    "Heston":       [f"{euro_heston:.2f}", f"{bar_heston:.2f}", f"{asian_heston:.2f}"],
    "LV vs BS":     [f"{(euro_lv/euro_bs-1)*100:+.2f}%", f"{(bar_lv/bar_bs-1)*100:+.2f}%",
                     f"{(asian_lv/asian_bs-1)*100:+.2f}%"],
    "Heston vs BS": [f"{(euro_heston/euro_bs-1)*100:+.2f}%", f"{(bar_heston/bar_bs-1)*100:+.2f}%",
                     f"{(asian_heston/asian_bs-1)*100:+.2f}%"],
})
display(df_res)


In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(3); w = 0.26
ax.bar(x-w, [euro_bs, bar_bs, asian_bs],   w, label="Black-Scholes", color="#6b7280", alpha=0.85)
ax.bar(x,   [euro_lv, bar_lv, asian_lv],   w, label="Local Vol",     color="#2563eb", alpha=0.85)
ax.bar(x+w, [euro_heston, bar_heston, asian_heston], w, label="Heston", color="#dc2626", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(["European Call (ATM)", "Up-and-Out Barrier", "Asian Call (Arith.)"], fontsize=11)
ax.set_ylabel("Option Price (CAC40 pts)")
ax.set_title(f"Exotic Pricing: BS vs Local Vol vs Heston  |  T={T_mc:.2f}y, K={K_atm}", fontweight="bold")
ax.legend()
plt.tight_layout(); plt.show()


## 10. Terminal Distribution & QQ Analysis

Comparing the risk-neutral terminal distributions of $S_T$ implied by the two models.
A heavier tail under Heston → different OTM put/call prices despite identical vanilla calibration.


In [ ]:
S_T_lv  = paths_lv[:, -1]
S_T_hes = paths_heston[:, -1]
from scipy.stats import norm as sp_norm

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(S_T_lv,  bins=80, alpha=0.6, color="#2563eb", density=True, label="Local Vol")
axes[0].hist(S_T_hes, bins=80, alpha=0.6, color="#dc2626", density=True, label="Heston")
axes[0].axvline(K_atm, color="black", ls="--", lw=1.5, label=f"ATM K={K_atm}")
axes[0].set_title(f"Terminal Distribution  S_T  (T={T_mc:.2f}y)", fontweight="bold")
axes[0].set_xlabel("$S_T$"); axes[0].set_ylabel("Density"); axes[0].legend()

ret_lv  = np.log(S_T_lv/S0)
ret_hes = np.log(S_T_hes/S0)
pp = np.linspace(0.01, 0.99, 300)
q_lv  = np.quantile(ret_lv,  pp)
q_hes = np.quantile(ret_hes, pp)
q_norm = sp_norm.ppf(pp, loc=np.mean(ret_lv), scale=np.std(ret_lv))

axes[1].plot(q_norm, q_lv,  "#2563eb", lw=2.5, label="Local Vol")
axes[1].plot(q_norm, q_hes, "#dc2626", lw=2.5, label="Heston")
axes[1].plot(q_norm, q_norm, "black",  lw=1.2, ls="--", label="Normal")
axes[1].set_title("Log-Return QQ Plot vs Normal\n(tails = model-implied skewness/kurtosis)", fontweight="bold")
axes[1].set_xlabel("Normal quantiles"); axes[1].set_ylabel("Model quantiles"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"\nDistribution moments (log-returns):")
print(f"{'':20} {'Mean':>10} {'Std':>10} {'Skew':>10} {'Kurt':>10}")
from scipy.stats import skew, kurtosis
for name, r in [("Local Vol", ret_lv), ("Heston", ret_hes)]:
    print(f"  {name:18} {np.mean(r):10.4f} {np.std(r):10.4f} {skew(r):10.4f} {kurtosis(r):10.4f}")


---
## Summary

| Step | Method | Key Result |
|------|--------|------------|
| Forward bootstrap | Call-put parity + bisection | 13 implied forwards extracted cleanly |
| Implied vol | Newton-Raphson + Halley (OTM) | 142 quotes, 0 calendar arb violations |
| SVI calibration | DE + L-BFGS-B (per slice) | Butterfly-free, smooth surface |
| Dupire local vol | Analytical from SVI derivatives | Avoids FD instability on sparse grid |
| Heston calibration | Multi-maturity LM, P1/P2 pricer | RMSE < 0.5 vol pts |
| MC: Local Vol | Euler-Maruyama, log-Euler | 30k paths, antithetic |
| MC: Heston | Quadratic Exponential (Andersen) | Unbiased variance discretisation |
| Exotic pricing | BS / LV / Heston | Barrier most sensitive to forward smile |

**Key takeaway:** Despite matching the same vanilla surface, Local Vol and Heston 
generate structurally different forward smiles — with downstream impact on barrier, 
Asian, and clique products. The model choice is a genuine pricing risk for exotics desks.
